<a href="https://colab.research.google.com/github/Atma453/DataGrokr/blob/main/DataGrokr_Week3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install pandas requests pytest -q

In [8]:
%%writefile etl.py

import requests
import pandas as pd


API_URL = "https://jsonplaceholder.typicode.com/posts"
OUTPUT_FILE = "etl_output.csv"


def extract_data(url):
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    return response.json()


def transform_data(data):
    df = pd.DataFrame(data)

    df = df[["userId", "id", "title", "body"]]

    df["title"] = df["title"].str.strip().str.title()
    df["body"] = df["body"].str.strip()

    df["title_length"] = df["title"].str.len()
    df["body_length"] = df["body"].str.len()

    df = df.drop_duplicates()

    return df


def load_data(df, output_file):
    df.to_csv(output_file, index=False)
    return output_file


def run_etl():
    print("Starting ETL Pipeline...")

    print("\n1. Extracting data from REST API...")
    data = extract_data(API_URL)
    print(f"Records fetched: {len(data)}")

    print("\n2. Transforming data...")
    df = transform_data(data)
    print(f"Records after transformation: {len(df)}")

    print("\n3. Loading data into CSV...")
    load_data(df, OUTPUT_FILE)
    print(f"CSV saved: {OUTPUT_FILE}")

    return df


if __name__ == "__main__":
    df = run_etl()
    print("\nPreview:")
    print(df.head())

Writing etl.py


In [9]:
from etl import run_etl

df = run_etl()

Starting ETL Pipeline...

1. Extracting data from REST API...
Records fetched: 100

2. Transforming data...
Records after transformation: 100

3. Loading data into CSV...
CSV saved: etl_output.csv


In [10]:
%%writefile test_etl.py

import pandas as pd
from etl import extract_data, transform_data, load_data


def test_transform_data_returns_dataframe():
    data = [
        {
            "userId": 1,
            "id": 1,
            "title": "hello world",
            "body": "this is a test"
        }
    ]

    df = transform_data(data)

    assert isinstance(df, pd.DataFrame)


def test_transform_data_columns():
    data = [
        {
            "userId": 1,
            "id": 1,
            "title": "hello world",
            "body": "this is a test"
        }
    ]

    df = transform_data(data)

    expected_columns = [
        "userId",
        "id",
        "title",
        "body",
        "title_length",
        "body_length"
    ]

    assert list(df.columns) == expected_columns


def test_title_transformation():
    data = [
        {
            "userId": 1,
            "id": 1,
            "title": "hello world",
            "body": "test body"
        }
    ]

    df = transform_data(data)

    assert df.iloc[0]["title"] == "Hello World"


def test_body_transformation():
    data = [
        {
            "userId": 1,
            "id": 1,
            "title": "test",
            "body": "   test body   "
        }
    ]

    df = transform_data(data)

    assert df.iloc[0]["body"] == "test body"


def test_duplicate_removal():
    data = [
        {
            "userId": 1,
            "id": 1,
            "title": "test",
            "body": "body"
        },
        {
            "userId": 1,
            "id": 1,
            "title": "test",
            "body": "body"
        }
    ]

    df = transform_data(data)

    assert len(df) == 1


def test_length_columns():
    data = [
        {
            "userId": 1,
            "id": 1,
            "title": "hello",
            "body": "testing"
        }
    ]

    df = transform_data(data)

    assert df.iloc[0]["title_length"] == 5
    assert df.iloc[0]["body_length"] == 7


def test_load_data(tmp_path):
    data = [
        {
            "userId": 1,
            "id": 1,
            "title": "hello",
            "body": "testing"
        }
    ]

    df = transform_data(data)

    output_file = tmp_path / "test_output.csv"

    load_data(df, output_file)

    assert output_file.exists()

    loaded_df = pd.read_csv(output_file)

    assert len(loaded_df) == 1

Overwriting test_etl.py


In [11]:
!pytest -v test_etl.py

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: typeguard-4.6.0, anyio-4.15.1, langsmith-0.12.5
collected 7 items                                                              

test_etl.py::test_transform_data_returns_dataframe PASSED                [ 14%]
test_etl.py::test_transform_data_columns PASSED                          [ 28%]
test_etl.py::test_title_transformation PASSED                            [ 42%]
test_etl.py::test_body_transformation PASSED                             [ 57%]
test_etl.py::test_duplicate_removal PASSED                               [ 71%]
test_etl.py::test_length_columns PASSED                                  [ 85%]
test_etl.py::test_load_data PASSED                                       [100%]

============================== 7 passed in 0.53s ===============================


In [12]:
import pandas as pd

result = pd.read_csv("etl_output.csv")

print("ETL Output:")
display(result.head(10))

print("\nShape:", result.shape)
print("\nColumns:")
print(result.columns.tolist())

ETL Output:


,userId,id,title,body,title_length,body_length
0,1,1,Sunt Aut Facere Repellat Provident Occaecati E...,quia et suscipit\nsuscipit recusandae consequu...,74,158
1,1,2,Qui Est Esse,est rerum tempore vitae\nsequi sint nihil repr...,12,206
2,1,3,Ea Molestias Quasi Exercitationem Repellat Qui...,et iusto sed quo iure\nvoluptatem occaecati om...,59,164
3,1,4,Eum Et Est Occaecati,ullam et saepe reiciendis voluptatem adipisci\...,20,190
4,1,5,Nesciunt Quas Odio,repudiandae veniam quaerat sunt sed\nalias aut...,18,147
5,1,6,Dolorem Eum Magni Eos Aperiam Quia,ut aspernatur corporis harum nihil quis provid...,34,194
6,1,7,Magnam Facilis Autem,dolore placeat quibusdam ea quo vitae\nmagni q...,20,145
7,1,8,Dolorem Dolore Est Ipsam,dignissimos aperiam dolorem qui eum\nfacilis q...,24,176
8,1,9,Nesciunt Iure Omnis Dolorem Tempora Et Accusan...,consectetur animi nesciunt iure dolore\nenim q...,50,136
9,1,10,Optio Molestias Id Quia Eum,quo et expedita modi cum officia vel magni\ndo...,27,129



Shape: (100, 6)

Columns:
['userId', 'id', 'title', 'body', 'title_length', 'body_length']


In [13]:
from google.colab import files

files.download("etl_output.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>